# SORT1 rs12740374 — cell-type screen

Reproduces the MCP `discover_variant_cell_types` call. Screens all
DNASE/ATAC tracks on **alphagenome**, ranks cell types by
variant effect magnitude (weighted by alt activity), then runs a
full multi-layer report per top cell type.


In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.discovery import discover_and_report

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
oracle = chorus.create_oracle(
    oracle_name='alphagenome',
    use_environment=True,
)
oracle.load_pretrained_model()

In [ ]:
# Inputs.
oracle_name = 'alphagenome'
variant_position = 'chr1:109274968'
alleles = ['G', 'T']
gene_name = 'SORT1'
top_n = 5
min_effect = 0.15
ranking_metric = "alt_x_abs_effect"
min_ref_value = 0.0

In [ ]:
# Two-stage discovery: screen → full report per top cell type.
# Writes one HTML per top cell type into the walkthrough dir.
normalizer = get_normalizer(oracle_name=oracle_name)
result = discover_and_report(
    oracle=oracle,
    variant_position=variant_position,
    alleles=alleles,
    gene_name=gene_name,
    top_n=top_n,
    min_effect=min_effect,
    output_path=str(WALKTHROUGH_DIR),
    normalizer=normalizer,
    igv_raw=False,
    oracle_name=oracle_name,
    user_prompt=None,
    tool_name="discover_variant_cell_types",
    ranking_metric=ranking_metric,
    min_ref_value=min_ref_value,
)

In [ ]:
# Save the screening summary (cell-type ranking + per-cell-type report dicts).
serializable = {
    "hits": result["hits"],
    "reports": {
        ct: rep.to_dict() for ct, rep in result.get("reports", {}).items()
    },
}
WALKTHROUGH_DIR.joinpath("discovery_summary.json").write_text(
    json.dumps(serializable, indent=2, default=str)
)
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(serializable, indent=2, default=str)
)
# Markdown summary lists the top hits.
_md_lines = [
    f"# Cell-type screen — {gene_name} {variant_position}",
    "",
    f"Top {top_n} cell types ranked by {ranking_metric}:",
    "",
    "| Cell type | Effect | |Effect| | Best track | n tracks |",
    "|---|---|---|---|---|",
]
for h in result["hits"]:
    _md_lines.append(
        f"| {h['cell_type']} | {h['effect']:+.3f} | {h['abs_effect']:.3f} | {h['best_track']} | {h['n_tracks']} |"
    )
WALKTHROUGH_DIR.joinpath("example_output.md").write_text("\n".join(_md_lines))
print(f"Wrote artifacts to {WALKTHROUGH_DIR}")

## What this notebook produced

- `discovery_summary.json` / `example_output.json` — cell-type ranking + per-cell-type report dicts
- `example_output.md` — markdown table of top cell types
- One HTML per top cell type (e.g. `chr*_*_<celltype>_report.html`)
